# Entity Extraction Evaluators Testing Notebook

This notebook demonstrates the usage of the modular evaluators for entity extraction quality assessment.

## Available Evaluators:
1. **ExtractionCorrectnessEvaluator**: Checks if extracted values are present in OCR text using fuzzy matching
2. **ExtractionCompletenessEvaluator**: Uses Azure OpenAI to assess if extracted values are relevant and complete

## Import Required Libraries

Import the evaluator classes and models from the evaluators package.

In [1]:
import sys
import os

# Add parent directory to path
sys.path.append(os.path.abspath('..'))

from evaluators.correctness_evaluator import ExtractionCorrectnessEvaluator
from evaluators.completeness_evaluator import ExtractionCompletenessEvaluator

print("✓ Evaluators imported successfully")

✓ Evaluators imported successfully


## 1. Extraction Correctness Evaluator

The **ExtractionCorrectnessEvaluator** uses fuzzy string matching (token set ratio) to determine if extracted values are present in the OCR text from Azure Document Intelligence.

### Key Features:
- Returns fuzzy similarity score (0-1)
- Marks extraction as correct only when fuzzy_score == 1.0
- Handles text normalization and cleaning automatically
- No threshold configuration needed

### Example 1.1: Basic Correctness Evaluation

Initialize the evaluator and test it with sample OCR text and extracted entities.

In [2]:
# Sample OCR text from Azure Document Intelligence
ocr_text = """
COMMERCIAL INVOICE

Invoice Number: INV-2024-00123
Invoice Date: January 15, 2024

Seller Information:
Company Name: Microsoft Corporation
Address: One Microsoft Way, Redmond, WA 98052
Tax ID: 91-1144442

Buyer Information:
Company: HSBC Bank
Contact: John Smith
Email: john.smith@hsbc.com

Line Items:
1. Software License - Enterprise    $25,000.00
2. Support Services - Annual        $5,000.00

Subtotal: $30,000.00
Tax (10%): $3,000.00
Total Amount Due: $33,000.00

Payment Terms: Net 30 Days
"""

# Initialize the correctness evaluator
correctness_eval = ExtractionCorrectnessEvaluator()

# Test with correctly extracted values
test_fields = [
    ("invoice_number", "INV-2024-00123"),
    ("seller_name", "Microsoft Corporation"),
    ("buyer_name", "HSBC Bank"),
    ("total_amount", "$33,000.00"),
    ("invoice_date", "January 15, 2024"),
]

print("=" * 80)
print("EXTRACTION CORRECTNESS EVALUATION RESULTS")
print("=" * 80)

for field_name, extracted_value in test_fields:
    result = correctness_eval.evaluate_field(
        field_name=field_name,
        extracted_value=extracted_value,
        source_text=ocr_text
    )
    
    status = "✓ CORRECT" if result.metadata['extraction_correct'] else "✗ INCORRECT"
    print(f"\nField: {field_name}")
    print(f"  Extracted Value: {extracted_value}")
    print(f"  Fuzzy Score: {result.metadata['fuzzy_score']:.2f}")
    print(f"  Status: {status}")

EXTRACTION CORRECTNESS EVALUATION RESULTS

Field: invoice_number
  Extracted Value: INV-2024-00123
  Fuzzy Score: 1.00
  Status: ✓ CORRECT

Field: seller_name
  Extracted Value: Microsoft Corporation
  Fuzzy Score: 1.00
  Status: ✓ CORRECT

Field: buyer_name
  Extracted Value: HSBC Bank
  Fuzzy Score: 1.00
  Status: ✓ CORRECT

Field: total_amount
  Extracted Value: $33,000.00
  Fuzzy Score: 1.00
  Status: ✓ CORRECT

Field: invoice_date
  Extracted Value: January 15, 2024
  Fuzzy Score: 1.00
  Status: ✓ CORRECT


### Example 1.2: Testing Incorrect Extractions

Test the evaluator with values that are NOT present in the OCR text to see how it handles incorrect extractions.

In [3]:
# Test with incorrect/missing values
incorrect_fields = [
    ("invoice_number", "INV-2024-99999"),  # Wrong number
    ("seller_name", "Apple Inc."),  # Wrong company
    ("total_amount", "$50,000.00"),  # Wrong amount
    ("payment_method", "Credit Card"),  # Not in document
]

print("\n" + "=" * 80)
print("TESTING INCORRECT EXTRACTIONS")
print("=" * 80)

for field_name, extracted_value in incorrect_fields:
    result = correctness_eval.evaluate_field(
        field_name=field_name,
        extracted_value=extracted_value,
        source_text=ocr_text
    )
    
    status = "✓ CORRECT" if result.metadata['extraction_correct'] else "✗ INCORRECT"
    print(f"\nField: {field_name}")
    print(f"  Extracted Value: {extracted_value}")
    print(f"  Fuzzy Score: {result.metadata['fuzzy_score']:.3f}")
    print(f"  Status: {status}")
    if 'reason' in result.metadata:
        print(f"  Reason: {result.metadata['reason']}")


TESTING INCORRECT EXTRACTIONS

Field: invoice_number
  Extracted Value: INV-2024-99999
  Fuzzy Score: 0.039
  Status: ✗ INCORRECT

Field: seller_name
  Extracted Value: Apple Inc.
  Fuzzy Score: 0.044
  Status: ✗ INCORRECT

Field: total_amount
  Extracted Value: $50,000.00
  Fuzzy Score: 0.044
  Status: ✗ INCORRECT

Field: payment_method
  Extracted Value: Credit Card
  Fuzzy Score: 0.044
  Status: ✗ INCORRECT


### Example 1.3: Batch Evaluation

Evaluate multiple fields at once using the `evaluate_batch` method for efficiency.

In [4]:
# Prepare batch data
batch_fields = [
    {"field_name": "invoice_number", "value": "INV-2024-00123"},
    {"field_name": "invoice_date", "value": "January 15, 2024"},
    {"field_name": "seller_name", "value": "Microsoft Corporation"},
    {"field_name": "buyer_name", "value": "HSBC Bank"},
    {"field_name": "subtotal", "value": "$30,000.00"},
    {"field_name": "tax", "value": "$3,000.00"},
    {"field_name": "total", "value": "$33,000.00"},
]

# Batch evaluate
results = correctness_eval.evaluate_batch(batch_fields, ocr_text)

print("\n" + "=" * 80)
print("BATCH EVALUATION RESULTS")
print("=" * 80)
print(f"\n{'Field Name':<20} {'Fuzzy Score':<15} {'Status':<15}")
print("-" * 80)

for result in results:
    status = "✓ CORRECT" if result.metadata['extraction_correct'] else "✗ INCORRECT"
    print(f"{result.field_name:<20} {result.metadata['fuzzy_score']:<15.3f} {status:<15}")


BATCH EVALUATION RESULTS

Field Name           Fuzzy Score     Status         
--------------------------------------------------------------------------------
invoice_number       1.000           ✓ CORRECT      
invoice_date         1.000           ✓ CORRECT      
seller_name          1.000           ✓ CORRECT      
buyer_name           1.000           ✓ CORRECT      
subtotal             1.000           ✓ CORRECT      
tax                  1.000           ✓ CORRECT      
total                1.000           ✓ CORRECT      


## 2. Extraction Completeness Evaluator

The **ExtractionCompletenessEvaluator** uses Azure OpenAI (GPT-4) to assess if extracted values are:
1. **Relevant** to the field name
2. **Complete** with all necessary information from the source

### Scoring Logic:
- **Relevance**: 50% of score (0.5 if relevant, 0.0 if not)
- **Completeness**: 50% of score
  - Complete: +0.5
  - 1 missing item: +0.3
  - 2 missing items: +0.2
  - 3+ missing items: +0.1

**Note**: This evaluator requires Azure OpenAI credentials and will make API calls.

### Example 2.1: Initialize Completeness Evaluator

Set up the Azure OpenAI client with your credentials. Update the endpoint and deployment name below.

In [ ]:
# Azure OpenAI Configuration
# TODO: Update these with your actual Azure OpenAI credentials

AZURE_OPENAI_ENDPOINT = "https://your-endpoint.openai.azure.com/"
DEPLOYMENT_NAME = "gpt-4-vision"
API_VERSION = "2024-08-01-preview"

# Initialize the completeness evaluator
# Uncomment when you have valid credentials
"""
from azure.identity import DefaultAzureCredential

completeness_eval = ExtractionCompletenessEvaluator(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    deployment_name=DEPLOYMENT_NAME,
    api_version=API_VERSION,
    credential=DefaultAzureCredential()
)
print("✓ Completeness evaluator initialized")
"""

print("⚠️  Completeness evaluator requires Azure OpenAI credentials")
print("   Update the configuration above and uncomment the initialization code")

### Example 2.2: Evaluate Completeness (Demo Code)

This demonstrates how to use the completeness evaluator once Azure OpenAI is configured.

In [ ]:
# Example completeness evaluation (requires Azure OpenAI setup)
"""
# Test different levels of completeness
test_cases = [
    ("seller_address", "One Microsoft Way, Redmond, WA 98052"),  # Complete
    ("seller_address", "Redmond, WA"),  # Incomplete - missing street
    ("invoice_amount", "$33,000"),  # Incomplete - missing cents
    ("invoice_amount", "$33,000.00"),  # Complete
    ("contact_info", "john.smith@hsbc.com"),  # Incomplete - missing name/phone
]

print("=" * 80)
print("COMPLETENESS EVALUATION RESULTS")
print("=" * 80)

for field_name, extracted_value in test_cases:
    result = completeness_eval.evaluate_field(
        field_name=field_name,
        extracted_value=extracted_value,
        source_text=ocr_text
    )
    
    print(f"\\nField: {field_name}")
    print(f"  Extracted Value: {extracted_value}")
    print(f"  Score: {result.score:.2f}")
    print(f"  Relevant: {result.metadata['is_relevant']}")
    print(f"  Complete: {result.metadata['is_complete']}")
    print(f"  Missing Info: {result.metadata['missing_info']}")
    print(f"  Reasoning: {result.metadata['reasoning']}")
"""

print("💡 Uncomment the code above after configuring Azure OpenAI credentials")

## 3. Working with ExtractionResult Models

Demonstrate how to use evaluators with the actual `ExtractionResult` and `ExtractedField` models from the entity extraction system.

### Example 3.1: Import Extraction Models

Import the data models used by the entity extraction system.

In [ ]:
# Import extraction models
sys.path.append(os.path.abspath('../../..'))

from entity_extraction.models.extraction import ExtractionResult, ExtractedField
from entity_extraction.models.citation import Citation

print("✓ Extraction models imported successfully")

### Example 3.2: Create Sample ExtractionResult

Create a mock extraction result with multiple fields to evaluate.

In [ ]:
# Create a sample ExtractionResult
extraction_result = ExtractionResult(
    fields=[
        ExtractedField(
            field_name="invoice_number",
            value="INV-2024-00123",
            value_type="string",
            confidence=0.98,
            citations=[
                Citation(
                    type="bounding_box",
                    page=1,
                    bbox={"x": 0.1, "y": 0.1, "width": 0.2, "height": 0.02},
                    text_snippet="Invoice Number: INV-2024-00123"
                )
            ],
            needs_review=False,
            model_source="gpt-4-vision"
        ),
        ExtractedField(
            field_name="total_amount",
            value="$33,000.00",
            value_type="currency",
            confidence=0.95,
            citations=[
                Citation(
                    type="bounding_box",
                    page=1,
                    bbox={"x": 0.1, "y": 0.8, "width": 0.15, "height": 0.02},
                    text_snippet="Total Amount Due: $33,000.00"
                )
            ],
            needs_review=False,
            model_source="gpt-4-vision"
        ),
        ExtractedField(
            field_name="seller_name",
            value="Microsoft Corporation",
            value_type="string",
            confidence=0.92,
            citations=[
                Citation(
                    type="bounding_box",
                    page=1,
                    bbox={"x": 0.1, "y": 0.3, "width": 0.25, "height": 0.02},
                    text_snippet="Company Name: Microsoft Corporation"
                )
            ],
            needs_review=False,
            model_source="gpt-4-vision"
        ),
    ]
)

print(f"✓ Created ExtractionResult with {len(extraction_result.fields)} fields")

### Example 3.3: Evaluate ExtractionResult Fields

Iterate through the extracted fields and evaluate each one against the OCR text.

In [ ]:
print("=" * 80)
print("EVALUATING EXTRACTION RESULT")
print("=" * 80)

for field in extraction_result.fields:
    # Evaluate correctness
    correctness_result = correctness_eval.evaluate_field(
        field_name=field.field_name,
        extracted_value=field.value,
        source_text=ocr_text
    )
    
    print(f"\n{'─' * 80}")
    print(f"Field: {field.field_name}")
    print(f"{'─' * 80}")
    print(f"  Extracted Value: {field.value}")
    print(f"  Model Confidence: {field.confidence:.2f}")
    print(f"  Model Source: {field.model_source}")
    print(f"\n  Correctness Evaluation:")
    print(f"    Fuzzy Score: {correctness_result.metadata['fuzzy_score']:.3f}")
    print(f"    Extraction Correct: {correctness_result.metadata['extraction_correct']}")
    
    # Show citations if available
    if field.citations:
        print(f"\n  Citations:")
        for i, citation in enumerate(field.citations, 1):
            print(f"    {i}. Page {citation.page}: \"{citation.text_snippet}\"")

## 4. Summary Statistics

Calculate aggregate statistics across all evaluated fields.

In [ ]:
# Calculate summary statistics
all_results = []

for field in extraction_result.fields:
    result = correctness_eval.evaluate_field(
        field_name=field.field_name,
        extracted_value=field.value,
        source_text=ocr_text
    )
    all_results.append(result)

# Compute metrics
total_fields = len(all_results)
correct_extractions = sum(1 for r in all_results if r.metadata['extraction_correct'])
avg_fuzzy_score = sum(r.metadata['fuzzy_score'] for r in all_results) / total_fields
avg_confidence = sum(f.confidence for f in extraction_result.fields) / total_fields

print("\n" + "=" * 80)
print("EVALUATION SUMMARY")
print("=" * 80)
print(f"\nTotal Fields Evaluated: {total_fields}")
print(f"Correct Extractions: {correct_extractions} ({correct_extractions/total_fields*100:.1f}%)")
print(f"Incorrect Extractions: {total_fields - correct_extractions} ({(total_fields-correct_extractions)/total_fields*100:.1f}%)")
print(f"\nAverage Fuzzy Score: {avg_fuzzy_score:.3f}")
print(f"Average Model Confidence: {avg_confidence:.3f}")

# Correlation analysis
print(f"\n{'Field':<20} {'Model Conf':<15} {'Fuzzy Score':<15} {'Correct':<10}")
print("-" * 80)
for field, result in zip(extraction_result.fields, all_results):
    status = "✓" if result.metadata['extraction_correct'] else "✗"
    print(f"{field.field_name:<20} {field.confidence:<15.3f} {result.metadata['fuzzy_score']:<15.3f} {status:<10}")

## 5. Next Steps

### Using Evaluators in Production:

1. **Correctness Evaluation**: Run on all extracted fields to verify they exist in OCR text
2. **Completeness Evaluation**: Run on fields that need human-like assessment (requires Azure OpenAI)
3. **Thresholding**: Set confidence thresholds based on fuzzy scores
4. **Review Flagging**: Flag fields with low fuzzy scores for human review

### Integration Pattern:

```python
# After extraction
for field in extraction_result.fields:
    # Check correctness
    correctness = correctness_eval.evaluate_field(
        field.field_name, field.value, ocr_text
    )
    
    # Flag for review if not correct
    if not correctness.metadata['extraction_correct']:
        field.needs_review = True
        field.review_reason = f"Low fuzzy score: {correctness.metadata['fuzzy_score']:.3f}"
```